# Expected vs applied scholarship
### Beca teórica vs beca aplicada

**Executed public case study · completely synthetic · period 2030-09**

A benefit may be applied even when eligibility is not met, missed when due, or recorded without a program. This control derives a theoretical benefit from enrollment and billed base, compares the applied amount, and makes attribution gaps visible.

Un beneficio puede aplicarse sin elegibilidad, omitirse cuando corresponde o registrarse sin carrera. Este control deriva la beca teórica desde inscripciones y base facturada, compara lo aplicado y hace visibles los problemas de atribución.

[Case study / caso](../docs/cases/03_scholarship.md) · [Español](../docs/cases/03_scholarship.es.md)

## 1. Universe and contracts / Universo y contratos

enrollments.csv, billed.csv, scholarship_rules.csv, scholarships.csv and scholarship_exceptions.csv. Event inputs have unique row IDs. Rules are unique by selected period/program. Applied benefit may have a missing program, deliberately routed outside keyed detail.

The comparison grain is **student + program after selecting period**. Every event must have a unique row ID. Currency values are fictional. No Oracle connection or environment secret is read.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "data" / "synthetic").is_dir())
sys.path.insert(0, str(root / "src"))
from financial_controls.demo import load_inputs
from financial_controls.contracts import scope, unique, KEYS
from financial_controls.money import money_column, format_cents
from financial_controls.reconciliation import reconcile, summarize
from financial_controls.controls import enrollment_control, billing_control, scholarship_control, units
from financial_controls.reporting import summarize_review, display_detail

period = "2030-09"
data = load_inputs(root)

In [2]:
manifest = pd.DataFrame([
    {"source": name, "rows": len(data[name]), "columns": len(data[name].columns)}
    for name in ["enrollments","billed","scholarships","scholarship_rules","scholarship_exceptions"]
])
display(manifest)
print("enrollments")
display(data["enrollments"])

print("billed")
display(data["billed"])

print("scholarships")
display(data["scholarships"])

print("scholarship_rules")
display(data["scholarship_rules"])

print("scholarship_exceptions")
display(data["scholarship_exceptions"])

,source,rows,columns
0,enrollments,10,5
1,billed,9,6
2,scholarships,6,5
3,scholarship_rules,2,4
4,scholarship_exceptions,2,7


enrollments


,row_id,student_ref,program_ref,period,chargeable
0,SYN-E01,SYN-S001,SYN-A,2030-09,Y
1,SYN-E02,SYN-S001,SYN-A,2030-09,Y
2,SYN-E03,SYN-S001,SYN-B,2030-09,Y
3,SYN-E04,SYN-S002,SYN-A,2030-09,Y
4,SYN-E05,SYN-S002,SYN-A,2030-09,N
5,SYN-E06,SYN-S003,SYN-B,2030-09,Y
6,SYN-E07,SYN-S003,SYN-B,2030-09,Y
7,SYN-E08,SYN-S004,SYN-A,2030-09,Y
8,SYN-E09,SYN-S005,SYN-A,2030-09,Y
9,SYN-E10,SYN-S007,SYN-A,2030-09,Y


billed


,row_id,student_ref,program_ref,period,amount,manual_adjustment
0,SYN-B01,SYN-S001,SYN-A,2030-09,100.00,N
1,SYN-B02,SYN-S001,SYN-A,2030-09,100.00,N
2,SYN-B03,SYN-S001,SYN-B,2030-09,50.00,N
3,SYN-B04,SYN-S002,SYN-A,2030-09,100.00,N
4,SYN-B05,SYN-S003,SYN-B,2030-09,150.00,N
5,SYN-B06,SYN-S004,SYN-A,2030-09,100.00,N
6,SYN-B07,SYN-S004,SYN-A,2030-09,100.00,N
7,SYN-B08,SYN-S004,SYN-A,2030-09,100.00,Y
8,SYN-B09,SYN-S007,SYN-A,2030-09,90.00,Y


scholarships


,row_id,student_ref,program_ref,period,amount
0,SYN-SC01,SYN-S001,SYN-A,2030-09,20.00
1,SYN-SC02,SYN-S002,SYN-A,2030-09,5.00
2,SYN-SC03,SYN-S003,SYN-B,2030-09,20.00
3,SYN-SC04,SYN-S004,SYN-A,2030-09,10.00
4,SYN-SC05,SYN-S005,SYN-A,2030-09,7.00
5,SYN-SC06,SYN-S001,,2030-09,3.00


scholarship_rules


,period,program_ref,min_subjects,rate_bps
0,2030-09,SYN-A,2,1000
1,2030-09,SYN-B,2,1000


scholarship_exceptions


,row_id,student_ref,program_ref,period,rule,direction,amount
0,SYN-X01,SYN-S002,SYN-A,2030-09,SYNTHETIC_NON_CHARGEABLE_ALLOWANCE,-1,5.00
1,SYN-X02,SYN-S004,SYN-A,2030-09,SYNTHETIC_SPECIAL_BENEFIT,1,10.00


## 2. Normalize and aggregate / Normalizar y agregar

Keys stay strings; null keys are not converted to zero. Money is parsed from decimal strings and becomes integer cents before summing. Cardinality is declared after grouping.

In [3]:
rules = data["scholarship_rules"].loc[data["scholarship_rules"]["period"].eq(period)].copy()
unique(rules, ["program_ref"])
display(rules)
enrollments = scope(data["enrollments"], period, ["chargeable"])
counts = enrollments.groupby(KEYS, as_index=False).size().rename(columns={"size": "subjects"})
billed = scope(data["billed"], period, ["amount"])
base = summarize(units(billed, money=True))
eligibility_inputs = base.merge(counts, on=KEYS, how="left", validate="one_to_one").merge(
    rules, on="program_ref", how="left", validate="many_to_one"
)
display(eligibility_inputs)

,period,program_ref,min_subjects,rate_bps
0,2030-09,SYN-A,2,1000
1,2030-09,SYN-B,2,1000


,student_ref,program_ref,lines,cents,subjects,period,min_subjects,rate_bps
0,SYN-S001,SYN-A,2,20000,2,2030-09,2,1000
1,SYN-S001,SYN-B,1,5000,1,2030-09,2,1000
2,SYN-S002,SYN-A,1,10000,2,2030-09,2,1000
3,SYN-S003,SYN-B,1,15000,2,2030-09,2,1000
4,SYN-S004,SYN-A,3,30000,1,2030-09,2,1000
5,SYN-S007,SYN-A,1,9000,1,2030-09,2,1000


## 3. A contract must fail when violated / Un contrato debe fallar

A duplicate event or rule must not multiply rows silently. The following deliberate synthetic error is caught and reported by category only.

In [4]:
bad_rules = pd.concat([rules, rules.iloc[[0]]], ignore_index=True)
try:
    unique(bad_rules, ["program_ref"])
except ValueError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("Duplicate rule should fail")

ValueError: Duplicate key violates cardinality contract


## 4. Compare, classify and explain / Comparar, clasificar y justificar

Eligibility constants and the two allowance amounts are fictional. Applied benefits use a positive convention; negative values fail rather than being silently made positive. The non-chargeable allowance and special benefit explain only their declared capacity, not the entire student's discrepancy.

Las constantes de elegibilidad y los dos importes justificantes son ficticios. El aplicado usa convención positiva; valores negativos fallan en lugar de convertirse silenciosamente a positivos. Las asignaciones por no arancelada y beneficio especial explican sólo su capacidad declarada.

Difference = actual − expected. Raw status and review status are separate.

In [5]:
result = scholarship_control(
    data["enrollments"], data["billed"], data["scholarships"],
    data["scholarship_rules"], data["scholarship_exceptions"], period=period
)
display(display_detail(result))
display(summarize_review(result))

,student_ref,program_ref,lines_expected,lines_actual,amount_expected,amount_actual,difference_lines,difference_amount,justified_amount,pending_amount,raw_status,review_status,presence
0,SYN-S001,SYN-A,0,0,20.00,20.00,0,0.00,0.00,0.00,OK,OK,both
1,SYN-S001,SYN-B,0,0,0.00,0.00,0,0.00,0.00,0.00,OK,OK,left_only
2,SYN-S002,SYN-A,0,0,10.00,5.00,0,-5.00,-5.00,0.00,DE MENOS,JUSTIFICADO,both
3,SYN-S003,SYN-B,0,0,15.00,20.00,0,5.00,0.00,5.00,DE MÁS,PENDIENTE,both
4,SYN-S004,SYN-A,0,0,0.00,10.00,0,10.00,10.00,0.00,DE MÁS,JUSTIFICADO,both
5,SYN-S005,SYN-A,0,0,0.00,7.00,0,7.00,0.00,7.00,DE MÁS,PENDIENTE,right_only
6,SYN-S007,SYN-A,0,0,0.00,0.00,0,0.00,0.00,0.00,OK,OK,left_only


,raw_status,review_status,cases,students,difference_lines,difference_cents,justified_cents,pending_cents
0,DE MENOS,JUSTIFICADO,1,1,0,-500,-500,0
1,DE MÁS,JUSTIFICADO,1,1,0,1000,1000,0
2,DE MÁS,PENDIENTE,2,2,0,1200,0,1200
3,OK,OK,3,2,0,0,0,0


In [6]:
display(result.detail[[*KEYS, "base_cents", "subjects", "min_subjects", "rate_bps",
                       "cents_expected", "cents_actual", "raw_status", "review_status"]])

,student_ref,program_ref,base_cents,subjects,min_subjects,rate_bps,cents_expected,cents_actual,raw_status,review_status
0,SYN-S001,SYN-A,20000.0,2.0,2.0,1000.0,2000,2000,OK,OK
1,SYN-S001,SYN-B,5000.0,1.0,2.0,1000.0,0,0,OK,OK
2,SYN-S002,SYN-A,10000.0,2.0,2.0,1000.0,1000,500,DE MENOS,JUSTIFICADO
3,SYN-S003,SYN-B,15000.0,2.0,2.0,1000.0,1500,2000,DE MÁS,PENDIENTE
4,SYN-S004,SYN-A,30000.0,1.0,2.0,1000.0,0,1000,DE MÁS,JUSTIFICADO
5,SYN-S005,SYN-A,NaN,NaN,NaN,NaN,0,700,DE MÁS,PENDIENTE
6,SYN-S007,SYN-A,9000.0,1.0,2.0,1000.0,0,0,OK,OK


## 5. Evidence is consumed, not assumed / Evidencia consumida, no supuesta

The allocation ledger preserves the evidence ID, rule, direction and consumed capacities. Original difference = signed justification + pending. A partial explanation leaves the case open.

In [7]:
display(result.allocations)
for measure in ["lines", "cents"]:
    assert (result.detail[f"difference_{measure}"] ==
            result.detail[f"justified_{measure}"] + result.detail[f"pending_{measure}"]).all()
display(result.detail.loc[result.detail["review_status"].eq("PENDIENTE"),
                          [*KEYS, "pending_lines", "pending_cents", "raw_status"]])

,student_ref,program_ref,evidence_id,rule,direction,lines,cents
0,SYN-S002,SYN-A,SYNTHETIC_SCHOLARSHIP_EXCEPTION:SYN-X01,SYNTHETIC_NON_CHARGEABLE_ALLOWANCE,-1,0,500
1,SYN-S004,SYN-A,SYNTHETIC_SCHOLARSHIP_EXCEPTION:SYN-X02,SYNTHETIC_SPECIAL_BENEFIT,1,0,1000


,student_ref,program_ref,pending_lines,pending_cents,raw_status
3,SYN-S003,SYN-B,0,500,DE MÁS
5,SYN-S005,SYN-A,0,700,DE MÁS


## 6. A failure totals alone would miss / Lo que el total oculta

65.00 applied − 45.00 theoretical = +20.00. Keyed differences explain +17.00 and the missing-program ledger holds +3.00. The complete bridge reconciles; keyed detail alone does not. Pending keyed net and absolute values are both 12.00, plus a separately reported unassigned 3.00.

65,00 aplicados − 45,00 teóricos = +20,00. El detalle explica +17,00 y el registro sin carrera +3,00. El puente completo concilia; el detalle solo no. El pendiente por clave neto y absoluto es 12,00, más 3,00 no asignados informados aparte.

In [8]:
display(result.unassigned)
display(result.detail.loc[result.detail["presence"].eq("right_only")])
student = result.detail.loc[result.detail["student_ref"].eq("SYN-S001")]
display(student[[*KEYS, "cents_expected", "cents_actual"]])
assert student["cents_actual"].sum() == 2000  # unassigned 300 cents not copied to either program
assert result.kpis["unassigned_cents"] == 300
assert not result.kpis["detail_reconciles"]
assert result.kpis["totals_bridge_reconciles"]

,row_id,student_ref,program_ref,period,amount,lines,cents
5,SYN-SC06,SYN-S001,<NA>,2030-09,3.00,0,300


,student_ref,program_ref,lines_expected,cents_expected,lines_actual,cents_actual,presence,difference_lines,pending_lines,difference_cents,pending_cents,mixed_direction,raw_status,justified_lines,justified_cents,review_status,subjects,min_subjects,rate_bps,base_cents
5,SYN-S005,SYN-A,0,0,0,700,right_only,0,0,700,700,False,DE MÁS,0,0,PENDIENTE,NaN,NaN,NaN,NaN


,student_ref,program_ref,cents_expected,cents_actual
0,SYN-S001,SYN-A,2000,2000
1,SYN-S001,SYN-B,0,0


## 7. Independent source-to-detail bridge / Puente independiente

Totals are calculated before joining. Unassigned amounts remain separate from keyed pending exposure. A complete bridge is not a declaration that all business exceptions are resolved.

In [9]:
bridge = pd.DataFrame([
    {"measure": m, "expected": result.kpis[f"expected_{m}"],
     "actual": result.kpis[f"actual_{m}"],
     "mother_difference": result.kpis[f"mother_difference_{m}"],
     "keyed_detail": result.kpis[f"detail_difference_{m}"],
     "unassigned": result.kpis[f"unassigned_{m}"]}
    for m in ["lines", "cents"]
])
display(bridge)
assert (bridge["mother_difference"] == bridge["keyed_detail"] + bridge["unassigned"]).all()
display(pd.DataFrame([{
    "pending_cases": result.kpis["pending_cases"],
    "pending_net_cents": result.kpis["pending_net_cents"],
    "pending_absolute_cents": result.kpis["pending_absolute_cents"],
    "unassigned_cents": result.kpis["unassigned_cents"],
    "control_clear": result.kpis["control_clear"]
}]))

,measure,expected,actual,mother_difference,keyed_detail,unassigned
0,lines,0,0,0,0,0
1,cents,4500,6500,2000,1700,300


,pending_cases,pending_net_cents,pending_absolute_cents,unassigned_cents,control_clear
0,2,1200,1200,300,False


## 8. Pipeline trace / Trazabilidad por etapas

The stage ledger records counts and check categories, never connection data or source records. Run the CLI with --verbose to see the same metadata in logging.

In [10]:
display(pd.DataFrame(result.trace))
assert result.trace[-1]["stage"] == "totals_bridge"
assert result.trace[-1]["check"] == "PASS"

,stage,rows,check
0,scope.enrollments,10,validated_keys_and_row_ids; excluded_rows=0
1,scope.billed,9,validated_keys_and_row_ids; excluded_rows=0
2,scope.applied,6,validated_keys_and_row_ids; excluded_rows=0
3,eligibility.rules,2,many_to_one;no_missing_configuration
4,eligibility.theory,6,one_to_one_counts;half_up_rate
5,route.unassigned,1,never_distribute_across_programs
6,scope.exceptions,2,validated_keys_and_row_ids; excluded_rows=0
7,input_contract,11,integer_measures;critical_keys;nonnegative_counts
8,aggregate.expected,6,unique_student_program
9,aggregate.actual,5,unique_student_program


## Review outputs and limits / Outputs y límites

Run `python -m financial_controls.demo --period 2030-09 --output artifacts` to export detail, summary, allocations, unassigned records, trace, KPIs and HTML.

The theoretical universe starts from billed base, not all potentially eligible students. The outer comparison catches applied-only benefits, but cannot infer a benefit for an eligible student with no billed base. No automatic allocation of missing-program benefits is attempted.

El universo teórico nace de la base facturada, no de todos los potencialmente elegibles. El outer conserva becas aplicadas sin base, pero no infiere una beca para alguien elegible sin base facturada. No se distribuyen automáticamente importes sin carrera.

### Interview / Entrevista
- Why is a missing-program benefit not added to every program? / ¿Por qué no sumar la beca sin carrera a todas las carreras?
- What is the difference between detail_reconciles and totals_bridge_reconciles? / ¿Qué diferencia hay entre detail_reconciles y totals_bridge_reconciles?
- Why reject a missing rule instead of assuming a default minimum? / ¿Por qué rechazar configuración faltante en lugar de asumir un mínimo?

No production processing, operational impact metric or confidential rule is reproduced.